# Read the val dataset

In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv('../data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1
val_labels = pd.read_csv('../data/dev_semeval_parids-labels.csv')["par_id"]  # extract the labels for val dataset 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 

# Read the prediction results: 
- baseline.txt (Roberta) 
- only_oversample.txt (Baseline Roberta + Oversample)
- roberta_ensemble.txt (Baseline Roberta + Oversample + Context)
- oversample_context_cr.txt (Baseline Roberta + Oversample + Context + Coreference resolution)
- bert_ensemble.txt (BERT + Oversample + Context)
- final.txt (Same as dev.txt, ensembles) 

In [2]:
with open("baseline.txt", "r") as f: 
    y_pred_baseline = np.array([int(x) for x in f.read().split("\n")]) 

with open("only_oversample.txt", "r") as f: 
    y_pred_only_oversample = np.array([int(x) for x in f.read().split("\n")]) 

with open("roberta_ensemble.txt", "r") as f: 
    y_pred_roberta_ensemble = np.array([int(x) for x in f.read().split("\n")]) 

with open("oversample_context_cr.txt", "r") as f: 
    y_pred_oversample_context_cr = np.array([int(x) for x in f.read().split("\n")]) 

with open("bert_ensemble.txt", "r") as f: 
    y_pred_bert_ensemble = np.array([int(x) for x in f.read().split("\n")]) 

with open("final.txt", "r") as f: 
    y_pred_final = np.array([int(x) for x in f.read().split("\n")]) 

y_true = df_val["bool_labels"]

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preds = [y_pred_baseline, y_pred_only_oversample, y_pred_roberta_ensemble, y_pred_oversample_context_cr, y_pred_bert_ensemble, y_pred_final]
for pred in preds: 
    print("="*50)
    
    print(f"Accuracy: {accuracy_score(y_true, pred):.3f}")
    print(f"Precision: {precision_score(y_true, pred):.3f}")
    print(f"Recall: {recall_score(y_true, pred):.3f}")
    print(f"F1: {f1_score(y_true, pred):.3f}")


Accuracy: 0.920
Precision: 0.582
Recall: 0.573
F1: 0.577
Accuracy: 0.917
Precision: 0.559
Recall: 0.618
F1: 0.587
Accuracy: 0.915
Precision: 0.541
Recall: 0.724
F1: 0.619
Accuracy: 0.913
Precision: 0.536
Recall: 0.598
F1: 0.565
Accuracy: 0.918
Precision: 0.577
Recall: 0.528
F1: 0.551
Accuracy: 0.920
Precision: 0.561
Recall: 0.719
F1: 0.630


# Examples of tp, tn, fp, fn for each class
Evaluated using the final model

In [7]:
pd.set_option("display.max_colwidth", None) 

keywords = ['homeless', 'disabled', 'poor-families', 'in-need', 'women', 'refugee', 'migrant', 'hopeless', 'immigrant', 'vulnerable']

# Extract examples of TP, TN, FP, FN, for each keyword
for keyword in keywords: 
    print("="*50)
    print(f"Keyword: {keyword}")
    print("="*50)
    tp = df_val[(df_val["keyword"] == keyword) & (y_true == 1) & (y_pred_final == 1)]
    tn = df_val[(df_val["keyword"] == keyword) & (y_true == 0) & (y_pred_final == 0)]
    fp = df_val[(df_val["keyword"] == keyword) & (y_true == 0) & (y_pred_final == 1)]
    fn = df_val[(df_val["keyword"] == keyword) & (y_true == 1) & (y_pred_final == 0)]

    display(tp[["text", "bool_labels"]].head()) 
    display(tn[["text", "bool_labels"]].head())
    display(fp[["text", "bool_labels"]].head()) 
    display(fn[["text", "bool_labels"]].head())

Keyword: homeless


,text,bool_labels
7,"It can not be right to allow homes to sit empty while many struggle to find somewhere to live , others having to sleep rough on pavements during Christmas , hoping against hope , for some charity to provide shelter . The number left homeless and destitute is alarming not necessarily at Christmas ?",True
26,"On the eve of the World Refugee Day , UNHCR received information about three new shipwrecks in the Mediterranean . It is feared that at least 130 people were dead or missing . Whatever the unpredictable Donald Trump may be doing or undoing , all our religions tell us that it is our sacred duty to give shelter to the homeless and that is what we need to do , discarding the trump cards from hell .",True
30,"Government support to bring the Housing First programme to Wellington will make a real difference for the homeless , says National 's Wellington Central candidate Nicola Willis .",True
39,"Behind the initiative are Danish national Lars Jorgensen and his American wife Monica , who came to the Philippines as backpacking tourists 27 years ago and were moved by the sight of Manila 's young and homeless .",True
45,"The drama 's compelling nature derived from a thorough knowledge of its cast , more numerous and improbable than any soap opera 's . The bite came from underlying tragedy : lost wives , lost jobs , pennilessness , homelessness and death . I have never laughed so much as in Soho in the Eighties .",True


,text,bool_labels
180,Clashes left 300 villagers homeless in Khagrachhari,False
181,"He regretted that in 2014 , Boko Haram killed 6,644 people and rendered more than one million homeless , making it the world 's deadliest militant group , according to Global Terrorism Index .",False
194,The Prime Minister said that construction of affordable housing initiative would not only provide shelter to the homeless but would generate huge economic activity by providing millions of jobs and boosting all linked industries of the housing sector .,False
215,He further alleged that it is unthinkable that the government of an independent country can make people homeless by killing them in police firing and resorting to attacks and torching .,False
223,"Loach remains just as passionate about social injustice as when his TV play "" Cathy Come Home "" shocked viewers 50 years ago with its depiction of a slide into homelessness .",False


,text,bool_labels
326,"As leaders , we will personally support victims but we ask the government to also help . We are going to clear all victims ' hospital bills and we want to ensure that those left homeless receive shelter .",False
356,His friends at the Chevron want people to know he was n't just a faceless homeless person . He was their friend and their family .,False
446,"Allman Town resident Sonya Wilson ( second left ) , and one of her daughters ( fourth left ) hand out boxed lunches to a group of homeless people on King Street , downtown Kingston , on Thursday .",False
500,"Black grew up in the neighbourhood around the church , where he would eat Franklin 's cooking at lavish meals she provided for the community and the homeless each Thanksgiving and Christmas . "" She made the best oxtail soup , with that cornbread , and it was to die for , "" he recalled . "" It would be so much food that you would n't know what to do . """,False
634,"6 years ago she lost her husband -- he died from heart disease , and since then Wood lives alone with her children and a dog . When she learnt about Michael and Cory 's terrible life problems , she realised she had to help . This woman has a big kind heart . There were 2 empty rooms in her house , so she invited the homeless to live there . Many people might consider her action as a madness but Mel says that since her husband 's death , she does n't fear anything .",False


,text,bool_labels
0,"His present "" chambers "" may be quite humble , but Shiyani has the tiny space very neatly organized and clean . Many people pass him by but do not manage to see him , because the space is partially hidden behind trees , which gives him a relative privacy . "" There are many homeless sleeping around the station , "" Captain Xoli Mbele , from the nearby Johannesburg Central Police station said .",True
12,"It 's calculated that over 204,000 days of purpose-built residential accommodation to otherwise potentially homeless elderly men and women have been delivered by this personally driven altruistic act alone .",True
20,"Bombarded by schizophrenia , addiction and homelessness , you might say that Eoghan O'Driscoll has been to hell and back . But he is finding a new balance through painting . Interview : Michael Lanigan",True
24,Mum living in homeless shelter has ' nowhere to bring boys on Christmas day ' <h> ' panicked ',True
49,"She remembers how hard being homeless hit her . She remembers , too , how she would try and act as mad as possible to intimidate people and stop them from coming close to her . She would turn as she walked and make loud noises so that people would leave her alone . How did she manage not to lose sight of her future while she was struggling for survival ? "" I handled my situation of being homeless pretty easily , "" she says with a bit of distance . "" I had football to focus on . I wanted to represent my country and I was n't going to let anything get in the way . """,True


...

Keyword: vulnerable


,text,bool_labels
27,"It is written in Ecclessiastes in the Bible that ' WOE TO ANY NATION WHERE A SLAVE BECOMES KING ' This typifies the situation in Nigeria since the inception of that country . I weep for my country when I visited home in 2009 and toured my alma mater- ABSU ... the infrastructural decay reduced me to tears . No matter how much OIL is sold in the world market the money will not change the corrupt/ sorry situation in Nigeria . The only thing that will change it is REVOLUTION . I weep for the children , pregnant women , the sick , infirms and the vulnerables who are trapped in this diabolical hole called Nigeria . My advice to Nigerians ... ' GOD WILL NOT DO FOR MAN WHAT MAN CAN DO FOR HIMSELF ' Wake and take your destinies in your hands stop calling on God for simple things you can do , learn from Egypt , Tunisia and Libya experience .",True
32,"This is a new education programme introduced in this district to ensure that each vulnerable girl and boy in this district is educated , protected , respected and valued and grows up to turn the tide of poverty .",True
64,"The Turkish Ambassador to Tanzania Ali Davutoglu has called on civil society groups and Tanzanians in general to join hands in helping orphans and vulnerable children , saying such responsibilities should not be left for the government alone .",True
82,""" We need to grow the economy in a way that we are helping those that are struggling -- families that are going to difficult times , and those that are in vulnerable situations , "" he said .",True
88,""" We shall remember him for the immense contribution he made to the many vulnerable sectors of humanity ? women ? children ? orphans the disabled and refugees . """,True


,text,bool_labels
171,"Monetary policy The IMF points out that further monetary tightening would head off second-round effects of currently high inflation , rein in credit expansion and protect against potential capital outflows from further US interest rate hikes . It suggests other measures to slow credit growth , including raising the reserve requirement and employing macro-prudential instruments , such as broader use of limits on loan-to-value ratios in vulnerable sectors and a credit limit or increasing risk weights in the housing sector .",False
201,"Last week , the Standard revealed Essex County Council would completely withdraw One Support , which provides social care to around 1,600 elderly and vulnerable residents with physical and mental health problems , from the Maldon district at the end of March . Vulnerable residents fear they will not be able to cope once their carers stop visiting them .",False
212,"New World Development ( NWD , 00017 . HK ) owns several commercial projects near the harbourfront area , making it vulnerable to criticisms that it will profit from its proposal to redevelop the Tsim Sha Tsui promenade , Apple Daily reported on Friday .",False
214,"The highest points total ND can now finish with is 103 , while Canterbury will have no fewer than 105 , a tally that still leaves them vulnerable to Auckland on the final day of the season .",False
218,The men were among a group convicted at London 's Southwark Crown Court of a 5.0 million pound ( $7.0 million ) scheme to steal public cash intended to be used to train vulnerable youngsters .,False


,text,bool_labels
626,""" Economic uncertainty can not be an excuse to slow down our development efforts , "" he said . "" It is a reason to speed them up . By investing in the MDGs , we invest in global economic growth . By focusing on the needs of the most vulnerable , we lay the foundation for a more sustainable and prosperous tomorrow . """,False
663,""" Protecting the disadvantaged members of our community is a cardinal role of any government and my administration in partnership with religious leaders will carry out fresh mapping of vulnerable members of our society and use the data to draw up a deliberate plan to support them , "" he said .",False
670,""" The UN Security Council must stand up and act to support vulnerable Palestinian people at the time when they need their protection . """,False
809,"It 's easy to dismiss the peace and love message as corny and pass ? , but Powers is convincing when she speaks of "" valuing people over things "" , and her beliefs are proven later when I learn of her considerable financial support of Taking it to the Streets , a charity helping vulnerable homeless youths , of whom there are many . ( This is depressing given the torrent of wealth pouring into the city from nearby Silicon Valley . If the Summer of Love set out to end stark inequality in its own community , it appears to have failed , despite the efforts of people like Powers . )",False
853,"In another development , Eiremiokhae was also bestowed with a Humanitarian Award by the Trinity International University of Ambassadors also in Georgia for his works of charity and care for widows and the vulnerable .",False


,text,bool_labels
41,"He said : "" I think we can consider introducing a negligible telecom surcharge to be entirely borne by the initiator of a call . In order to protect the poor and vulnerable amongst us , we could structure it to only take effect after the third minute of talk .",True
68,News this month that Anglican Care spent $4 million buying a Christchurch site as a hub for vulnerable youths is a fine example of that community-minded approach . In years to come there may be an interesting contrast between a beautifully restored cathedral that is empty much of the time and a youth hub and revamped City Mission that are constantly busy .,True
100,"Balu , an honest , hard-working labourer , who was injured by army shelling about 1993 leading to partial deafness , had latterly resettled in Tellipalai . While waiting to cross the KKS Rd. , he was killed by a navy vehicle with , as I learn , defective brakes , driven by a man without a heavy vehicle licence . When development fails the most vulnerable and poor , we have lost our way . It is well to remember Tagore in his essay on Nationalism : "" ... speed comes to its end , the engagement loses its meaning and the hungry heart clamours for food , till at last she comes to the lowly reaper reaping his harvest in the sun . """,True
143,"Emmanuelle Riva in "" Amour , "" as a woman in her 80s whose mind and body deteriorate after a series of strokes , is even frailer , more vulnerable , and quite defenseless . The first time we saw Riva , in "" Hiroshima Mon Amour , "" we were struck by her sensual , yet patrician face . Now the sensuality is gone , but the dignity of that countenance remains .",True
1753,"After a tragic event during his previous life as an FBI agent , Sawyer lost half a leg and developed an aversion to guns . So instead of the usual muscle-bound , wise-cracking , gun-toting cliche he 's often played , here he 's a vulnerable , disabled family man caught up in a twisted extortion plot .",True


In [77]:
# print(tp["keyword"].value_counts())
# print(tn["keyword"].value_counts())
print(fp["keyword"].value_counts())
print(fn["keyword"].value_counts())

keyword
poor-families    20
hopeless         17
in-need          17
vulnerable       13
homeless         12
disabled         11
refugee           9
women             9
migrant           3
immigrant         1
Name: count, dtype: int64
keyword
poor-families    14
women             8
hopeless          7
homeless          7
disabled          6
vulnerable        5
migrant           3
refugee           3
immigrant         2
in-need           1
Name: count, dtype: int64


# Metrics for each keyword for each model

In [73]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# F1, accuracy, precision, recall
for f in [f1_score, accuracy_score, precision_score, recall_score]: 
    data = [] 
    for keyword in keywords: 
        row = [keyword, df_val[df_val["keyword"] == keyword].shape[0], df_val[(df_val["keyword"] == keyword) & (df_val["bool_labels"] == True)].shape[0]]
        for pred in preds: 
            row.append(round(f(df_val[df_val["keyword"] == keyword]["bool_labels"], pred[df_val["keyword"] == keyword]), 3)) 
        data.append(row) 

    row = ["OVERALL", df_val.shape[0], df_val[df_val["bool_labels"] == True].shape[0]]
    for pred in preds: 
        row.append(round(f(df_val["bool_labels"], pred), 3)) 
    data.append(row) 

    data.sort(key = lambda x: x[1], reverse=True)

    df_f1 = pd.DataFrame(data)
    df_f1.columns = ["Keyword", "Count", "True Count", "Roberta Baseline", "Oversample", "Roberta ensemble (Oversample+Context)", "Oversample+Context+CR", "Bert ensemble", "Final"]
    display(df_f1)

,Keyword,Count,True Count,Roberta Baseline,Oversample,Roberta ensemble (Oversample+Context),Oversample+Context+CR,Bert ensemble,Final
0,OVERALL,2093,199,0.577,0.587,0.619,0.565,0.551,0.630
1,women,233,14,0.286,0.538,0.400,0.538,0.300,0.414
2,in-need,226,33,0.761,0.730,0.762,0.734,0.711,0.780
3,immigrant,218,7,0.600,0.364,0.769,0.333,0.600,0.769
4,hopeless,217,26,0.509,0.557,0.594,0.492,0.623,0.613
5,homeless,212,29,0.613,0.613,0.677,0.567,0.561,0.698
6,vulnerable,209,20,0.537,0.591,0.625,0.537,0.471,0.625
7,migrant,206,5,0.500,0.600,0.500,0.600,0.444,0.400
8,disabled,194,14,0.615,0.414,0.471,0.560,0.615,0.485
9,poor-families,190,38,0.560,0.528,0.588,0.560,0.492,0.585


,Keyword,Count,True Count,Roberta Baseline,Oversample,Roberta ensemble (Oversample+Context),Oversample+Context+CR,Bert ensemble,Final
0,OVERALL,2093,199,0.920,0.917,0.915,0.913,0.918,0.920
1,women,233,14,0.936,0.948,0.923,0.948,0.940,0.927
2,in-need,226,33,0.925,0.912,0.912,0.907,0.903,0.920
3,immigrant,218,7,0.982,0.968,0.986,0.963,0.982,0.986
4,hopeless,217,26,0.876,0.876,0.880,0.848,0.894,0.889
5,homeless,212,29,0.887,0.887,0.906,0.877,0.882,0.910
6,vulnerable,209,20,0.909,0.914,0.914,0.909,0.914,0.914
7,migrant,206,5,0.981,0.981,0.971,0.981,0.976,0.971
8,disabled,194,14,0.948,0.912,0.907,0.943,0.948,0.912
9,poor-families,190,38,0.826,0.821,0.816,0.826,0.826,0.821


,Keyword,Count,True Count,Roberta Baseline,Oversample,Roberta ensemble (Oversample+Context),Oversample+Context+CR,Bert ensemble,Final
0,OVERALL,2093,199,0.582,0.559,0.541,0.536,0.577,0.561
1,women,233,14,0.429,0.583,0.375,0.583,0.500,0.400
2,in-need,226,33,0.711,0.659,0.627,0.630,0.628,0.653
3,immigrant,218,7,1.000,0.500,0.833,0.400,1.000,0.833
4,hopeless,217,26,0.483,0.486,0.500,0.410,0.543,0.528
5,homeless,212,29,0.576,0.576,0.636,0.548,0.571,0.647
6,vulnerable,209,20,0.524,0.542,0.536,0.524,0.571,0.536
7,migrant,206,5,0.667,0.600,0.429,0.600,0.500,0.400
8,disabled,194,14,0.667,0.400,0.400,0.636,0.667,0.421
9,poor-families,190,38,0.568,0.559,0.532,0.568,0.593,0.545


,Keyword,Count,True Count,Roberta Baseline,Oversample,Roberta ensemble (Oversample+Context),Oversample+Context+CR,Bert ensemble,Final
0,OVERALL,2093,199,0.573,0.618,0.724,0.598,0.528,0.719
1,women,233,14,0.214,0.500,0.429,0.500,0.214,0.429
2,in-need,226,33,0.818,0.818,0.970,0.879,0.818,0.970
3,immigrant,218,7,0.429,0.286,0.714,0.286,0.429,0.714
4,hopeless,217,26,0.538,0.654,0.731,0.615,0.731,0.731
5,homeless,212,29,0.655,0.655,0.724,0.586,0.552,0.759
6,vulnerable,209,20,0.550,0.650,0.750,0.550,0.400,0.750
7,migrant,206,5,0.400,0.600,0.600,0.600,0.400,0.400
8,disabled,194,14,0.571,0.429,0.571,0.500,0.571,0.571
9,poor-families,190,38,0.553,0.500,0.658,0.553,0.421,0.632


# Comparison with the baseline model

In [30]:
# Examples correctly classified for both models 
cc = df_val[(y_true == y_pred_final) & (y_true == y_pred_baseline)]
cw = df_val[(y_true == y_pred_final) & (y_true != y_pred_baseline)]
wc = df_val[(y_true != y_pred_final) & (y_true == y_pred_baseline)]
ww = df_val[(y_true != y_pred_final) & (y_true != y_pred_baseline)]

display(cc[["text", "bool_labels"]].sample(frac=1).head()) 
display(cw[["text", "bool_labels"]].sample(frac=1).head()) 
display(wc[["text", "bool_labels"]].sample(frac=1).head()) 
display(ww[["text", "bool_labels"]].sample(frac=1).head())

,text,bool_labels
904,"Barring a disabled list stint for veteran Brett Gardner , who is nursing a sore knee , Frazier likely will be up for only a couple of days before heading back to the minor leagues . He has been in four major league games this year and has played 42 in the minors .",False
1811,"Movements require a mobilising vision , commitment , organisation , struggle , feedback , and participatory decision-making . Otherwise , any progress will be sporadic , temporary and insufficient to overcome the political inertia . The hopeless response to constructive proposals will remain : who is listening ?",False
1644,Race Relations Commissioner Dame Susan Devoy is in Geneva and has asked a United Nations committee to urge the New Zealand government to initiate an inquiry into the physical and sexual abuse of children and disabled people held in state institutions . More&gt;&gt;,False
427,""" Our findings show that the large space requirements for the cheetah , coupled with the complex range of threats faced by the species in the wild mean that it is likely to be much more vulnerable to extinction than was previously thought , "" Durant said .",False
2008,""" The study finds that the Clean Power Plan will inflict severe and disproportionate economic burdens on poor families , especially minorities , "" said Alford in his prepared statement . "" The EPA 's proposed regulation for GHG greenhouse gas emissions from existing power plants is a slap in the face to poor and minority families .",False


,text,bool_labels
1460,TurkIt 's heartening to see that measures are being taken in Khyber Pakhtunkhwa ( KP ) to empower women and give them work opportunities . You ! takes a look ...,True
715,"The vast majority of girls and women caught in the exploitative global sex trade are not victims of kidnapping , like the Nigerian 276 abducted by Boko Haram , but rather of poverty . Human traffickers prey on poor families who do n't have access to education and are n't aware of their basic rights . Mired in grinding poverty , parents desperately take out loans on conditions they do n't understand , pledging their children on their debts .",True
471,Rulers of Nato states that are enthusiastic about propping up this strategic alliance financially seem to have no regard for the suffering of their people . Rising unemployment rates and falling living standards have spawned far-right and fascist groups that are attacking immigrants and pushing European societies towards anarchy .,False
1861,"BRITAIN 'S protracted campaign of budget cutting , started in 2010 by a government led by the Conservative Party , has delivered a monumental shift in British life . A wave of austerity has yielded a country that has grown accustomed to living with less , even as many measures of social well-being - crime rates , opioid addiction , infant mortality , childhood poverty and homelessness - point to a deteriorating quality of life .",False
1541,"Columns <h> Poor , pregnant and homeless",False


,text,bool_labels
1006,""" If government makes education free at all levels , it will help many of us from poor families . I am ready to return to school if I see someone who can train me . I am not happy that my madam 's children go to school and I only wash clothes , sell anything they give to me and do all the work in the house . """,False
1700,"Artist Proshanta Karmakar Buddha has created his own style of art that is both modern and unique . He longs for a world devoid of chaos , brutality and hopelessness . He anticipates a peaceful utopia where human empathy and cooperation transcend national boundaries . It may be a ' fool 's dream ' but a world without hope is not a world worth living in . The spirit of that hope echoes through his works . Till date , Buddha has put up 29 solo exhibitions and participated in at least 96 national and international group exhibitions .",False
2023,""" I think you will see President Trump being willing to give legal status to some of the illegal immigrants who are not bad hombres if he can get better border security and more robust legal immigration , "" he said . "" I may be wrong but I think he can fix this . "" <h> Different places , different issues",False
633,The dream of a social protection plan whose role is to protect the elderly and the disabled from extreme forms of poverty through monthly stipends is quickly becoming a reality in Kenya .,False
500,"Black grew up in the neighbourhood around the church , where he would eat Franklin 's cooking at lavish meals she provided for the community and the homeless each Thanksgiving and Christmas . "" She made the best oxtail soup , with that cornbread , and it was to die for , "" he recalled . "" It would be so much food that you would n't know what to do . """,False


,text,bool_labels
1096,"A happy day it was indeed when a 31 homeless street dogs found their forever homes in the loving arms of the kind children and adults who visited Embark 's ' Adopt A Dog Day ' at Lyceum International School , Nugegoda on the 15th of September . Students from grades 1 to 8 were invited to attend along with their families and friends . The organizers were delighted by the fact that all the puppies and adult dogs who had been put up for adoption found good homes and new families in record time . In fact , Embark ran out of dogs as the demand was so high , but this was just the first adoption day of many more to come at the Lyceum Schools , so everyone who missed out can find their forever friends at future adoption days .",False
1920,""" This definitive outcome is a testament to the foresight of those who launched the programme , believing that elimination was possible in one of the world 's most endemic countries . In human terms , these children will never have to worry about being disabled by lymphatic filariasis . """,False
446,"Allman Town resident Sonya Wilson ( second left ) , and one of her daughters ( fourth left ) hand out boxed lunches to a group of homeless people on King Street , downtown Kingston , on Thursday .",False
1,"Krueger recently harnessed that creativity to self-publish a book featuring the poems , artwork , photography and short stories of 16 ill or disabled artists from around the world . She hopes the book , which contains some of her own work as well , will show how talented disabled people can be .",True
9,"He depicts demonstrations by refugees at the border post , their catastrophic living conditions and the desperate attempt of several hundred to cross a river a few kilometres from the camp to get into Macedonia on 14 March 2016 .",True


In [31]:
# For examples that were previously wrong but now correct, inspect its keyword distribution 
cw["keyword"].value_counts() 

keyword
homeless         8
hopeless         8
poor-families    7
vulnerable       6
refugee          5
in-need          5
immigrant        3
women            3
disabled         2
migrant          1
Name: count, dtype: int64

# Effect of oversample model
Comparing oversample only to baseline

In [36]:
# Sentences where only oversample performs better than baseline
cw = df_val[(y_true == y_pred_only_oversample) & (y_true != y_pred_baseline)]
display(cw[["keyword", "text", "bool_labels"]].sample(frac=1).head()) 

# Keyword distribution
cw["keyword"].value_counts() 

,keyword,text,bool_labels
1897,hopeless,"Rather sad . Good set of pictures that tells a tale of survival , subsistence living , and hopeless nature of life in the tribal societies . Exploiting an unexpected geo-political bonanza is a temporary relief that is not sustainable . Education and economic development seem miles away .",True
715,poor-families,"The vast majority of girls and women caught in the exploitative global sex trade are not victims of kidnapping , like the Nigerian 276 abducted by Boko Haram , but rather of poverty . Human traffickers prey on poor families who do n't have access to education and are n't aware of their basic rights . Mired in grinding poverty , parents desperately take out loans on conditions they do n't understand , pledging their children on their debts .",True
2071,poor-families,"Desertification which affects Yunusari , Yusufari , Karasuwa , Machina , Geidam and Bursari local government areas is increasingly diminishing the space for agricultural activities and livestock development . For many years , Yobe state has been screaming and asking for help to deal with desertification because it is depriving many poor families of their means of livelihood and even shelter . One needs to just visit Tulo-Tulo in Yusufari Local Government Area to have a picture of the disaster that is making life more difficult every day for thousands of families .",False
153,refugee,"In Dublin , the Church of Ireland Archbishop Dr Michael Jackson reflected on the year drawing to a close and recalled the "" horrific conflagration at the halting site in Carrickmines "" and the death of Garda Tony Golden in Omeath , as well as the "" refugees fleeing from Syria "" and other parts of the Middle East and the "" forgotten peoples of Africa "" .",True
668,poor-families,"If the situation is worsening in the cities , it is likely that in rural and remote regions children are even more at risk , particularly in Balochistan and Sindh , where poverty is at its highest . No data is available on rural areas , but many families face a daily struggle to feed everyone and extra children can be seen as unaffordable . Though many Pakistani women would like to have access to family planning , the use of birth control methods is still very low for cultural reasons and abortion is illegal . One gynaecologist told IRIN "" the mothers themselves wish to save the children but they also see the economic struggle of their families in a time of growing inflation "" . It says something about the sheer desperation of poor families in Pakistan , that murdering infants is seen as the only option open to them .",False


keyword
homeless         7
poor-families    7
refugee          6
hopeless         6
women            5
vulnerable       4
in-need          3
migrant          2
disabled         1
Name: count, dtype: int64

# Effect of adding context
Compare oversample+context to oversample only

In [37]:
# Sentences where only oversample performs better than baseline
cw = df_val[(y_true == y_pred_roberta_ensemble) & (y_true != y_pred_only_oversample)]
display(cw[["keyword", "text", "bool_labels"]].sample(frac=1).head()) 

# Keyword distribution
cw["keyword"].value_counts() 

,keyword,text,bool_labels
1227,homeless,"More than 200 people , young and old , were being fed at a soup kitchen . Many were homeless and all of them had an urgent need for some food to try and ward off the effects of the bitter weather .",False
1580,homeless,""" He was not a bum or a homeless guy , "" added Sisson .",False
386,poor-families,"I am shocked , disgusted and dismayed at yet another police incident that is being mishandled . How many poor families have had their loved ones senselessly perish at the hands of highly paid , supposedly "" professional "" police officers and have had to fight against an unjust system that allows these officers to literally get away with murder unpunished. ?",True
116,poor-families,Real poverty of Britain : Shocking images of UK in the Sixties where poor really meant poor <h> THESE hard-hitting photographs offer a glimpse into the harrowing day-to-day for poor families living in Britain during the Sixties .,True
1036,poor-families,""" We have not been paid for the last two months , this money has assisted many poor families and we would like the governor not to abolish the program completely as it will affect many households , "" she told Sunday nation .",False


keyword
homeless         9
in-need          9
poor-families    8
disabled         6
vulnerable       5
hopeless         5
immigrant        4
refugee          2
women            2
migrant          1
Name: count, dtype: int64

# Effect of CR
Compare oversample+context+CR to oversample+context

In [44]:
# Sentences where the CR fails
wc = df_val[(y_true != y_pred_oversample_context_cr) & (y_true == y_pred_roberta_ensemble)]
display(wc[["keyword", "text", "bool_labels"]].head()) 


print(wc["text"].tolist())


,keyword,text,bool_labels
8,women,""" People do n't understand the hurt , people do n't understand the pain . I 've read about women with their children sleeping in cars , sleeping in hotel rooms and it 's criminal . If they 're lucky and they come across COPE Galway and the ladies in Osterley , then there 's hope . """,True
16,immigrant,"Sheepherding in America has always been an immigrant 's job , too dirty , too cold and too lonely for anyone with options .",True
18,in-need,"He said : "" We need improved security for civilians and aid workers and access to all those in need , but we must also build a bigger humanitarian muscle to provide for the suffering millions .",True
19,women,"She continued , "" I stepped away from hiding behind a fabricated version of myself . I no longer put actions behind my fears and insecurities . I made a choice to redirect my energy to be a catalyst for change . To create a channel for women to become the truest versions of themselves , along with me. """,True
33,disabled,"Haiti has legal protections for the disabled on paper , but the laws are poorly implemented . Disabled Haitians have few opportunities to work and too many youngsters with disabilities languish out of sight at home instead of going to school . Some impoverished parents abandon disabled kids outside state institutions or farm them out as domestic servants .",True


['" People do n\'t understand the hurt , people do n\'t understand the pain . I \'ve read about women with their children sleeping in cars , sleeping in hotel rooms and it \'s criminal . If they \'re lucky and they come across COPE Galway and the ladies in Osterley , then there \'s hope . "', "Sheepherding in America has always been an immigrant 's job , too dirty , too cold and too lonely for anyone with options .", 'He said : " We need improved security for civilians and aid workers and access to all those in need , but we must also build a bigger humanitarian muscle to provide for the suffering millions .', 'She continued , " I stepped away from hiding behind a fabricated version of myself . I no longer put actions behind my fears and insecurities . I made a choice to redirect my energy to be a catalyst for change . To create a channel for women to become the truest versions of themselves , along with me. "', 'Haiti has legal protections for the disabled on paper , but the laws are 

Their coreference resolution versions: 

- " People do n't understand the hurt , people do n't understand the pain . I 've read about women with women with their children sleeping in cars , sleeping in hotel rooms children sleeping in cars , sleeping in hotel rooms and it 's criminal . If women with their children sleeping in cars , sleeping in hotel rooms 're lucky and women with their children sleeping in cars , sleeping in hotel rooms come across COPE Galway and the ladies in Osterley , then there 's hope . "

- Sheepherding in America has always been an immigrant 's job , too dirty , too cold and too lonely for anyone with options .

- He said : " We need improved security for civilians and aid workers and access to all those in need , but We must also build a bigger humanitarian muscle to provide for the suffering millions .

- myself continued , " myself stepped away from hiding behind a fabricated version of myself . myself no longer put actions behind myself fears and insecurities . myself made a choice to redirect myself energy to be a catalyst for change . To create a channel for themselves to become the truest versions of themselves , along with myself. "

- Haiti has legal protections for the disabled on paper , but legal protections for the disabled on paper are poorly implemented . Disabled Haitians have few opportunities to work and too many youngsters with disabilities languish out of sight at home instead of going to school . Some impoverished parents abandon disabled kids outside state institutions or farm disabled kids out as domestic servants .

# Effect of Ensembles
Compare ensembles for roberta and bert

In [ ]:
# Sentences where ensemble correct, roberta wrong 
cw = df_val[(y_true == y_pred_final) & (y_true != y_pred_roberta_ensemble)]
display(cw[["keyword", "text", "bool_labels"]].sample(frac=1).head()) 
print(cw["bool_labels"].value_counts())

# Sentences where ensemble correct, bert wrong 
cw = df_val[(y_true == y_pred_final) & (y_true != y_pred_bert_ensemble)]
display(cw[["keyword", "text", "bool_labels"]].sample(frac=1).head()) 
print(cw["bool_labels"].value_counts())

# Roberta wrong on mostly false labels, meaning it tends to over-predict for true labels
# On the other hand, BERT tends to under-predict true labels. 

,keyword,text,bool_labels
1898,migrant,"( Bloomberg ) -- California Senator Kamala Harris charged that the Trump administration had committed "" crimes against humanity "" after meeting at a U.S. detention center on Friday with immigrant mothers who had been separated from their children .",False
1139,hopeless,""" It beggars belief this scheme has been cobbled together ten weeks from the election when for more than a year Bill English has preferred to write off young unemployed people as pretty damn hopeless and too drugged and lazy .",False
462,in-need,"So starting in 1964 and for almost a decade , the federal government poured at least some of its resources in the direction they should have been going all along : toward those who were most in need . Longstanding programs like Head Start , Legal Services , and the Job Corps were created . Medicaid was established . Poverty among seniors was significantly reduced by improvements in Social Security .",False
135,homeless,"With a mission to "" strive every day to create a safe haven where homeless women and children find stability and access to the basic needs of life "" , the Elfreeda Foundation launched its open shelter on the 11th of August 2017 . In attendance were dignitaries such as : the wife of the vice president of Nigeria , Her Excellency Dolapo Osinbajo ; the governor of Imo State , His Excellency Owelle Anayo Rochas Okorocha ( OON ) ; wife of the governor of Imo State , Nneoma Nkechi Rochas Okorocha ; wife of the governor of Enugu state , Monica Ugwuanyi ; the chief of staff of the Imo State Government , Honorable Uche Nwosu ; publisher of Genevieve Magazine , Betty Irabor amongst others .",True
287,women,"She added : "" I would also like to carefully point out that the issue was not her religious beliefs , but rather it is about choosing to treat men and women differently by shaking the hands of women but not men . """,False


bool_labels
False    12
True      1
Name: count, dtype: int64


,keyword,text,bool_labels
104,vulnerable,The Minister said a society 's measure of its humanity is how it treats its weakest and most vulnerable members .,True
110,refugee,"Bus driver Cathal Carroll asks if I 've heard the news this morning . I have n't . Four thousand souls have just been rescued from the waters of the Mediterranean , he says . All of them African refugees . All fleeing hunger and persecution in their native lands . What do I think about that ?",True
474,women,"I am a Jaffna Tamil residing in the USA and I see the Sri Lankan professionals over here , both the Tamils and Sinhalese , holding top positions in Research , Universities HighTech industries , Law , and Finance.But then looking at the sterile Arab countries all we see are Sri Lankan women working as maids and servents to the rags to riches Arabs.A self respecting society would n't allow its women to be transported as maids to foreign land .",False
1034,poor-families,"Once that is done , the family will work toward raising funds to build small homes for poor families and women in crisis .",False
73,disabled,""" In particular , the programmes to support blind and disabled golf impress me both as an avid golfer and a passionate believer in the "" power of sport "" , to bring people together and transform lives for the better , "" said Mr Key .",True


bool_labels
True     44
False    29
Name: count, dtype: int64
